# Social media donation lift prediction

Predict expected donation value attributable to a planned social post, using historical `social_media_posts.csv`.

- **Label**: `estimated_donation_value_php` (regression)
- **Features (pre-publish)**: content + posting metadata + (optional) boost budget
- **Leakage avoidance**: time-based split by `created_at`
- **Output**: `ml-outputs/social-donation-lift/predictions.csv` (holdout predictions)

In [ ]:
import os
import warnings
from dataclasses import dataclass

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')

DATA_DIR = '../lighthouse_csv_v7'
OUT_DIR = '../ml-outputs/social-donation-lift'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

label_col = 'estimated_donation_value_php'
date_col = 'created_at'

prepublish_categorical = [
    'platform', 'day_of_week', 'post_type', 'media_type',
    'has_call_to_action', 'call_to_action_type',
    'content_topic', 'sentiment_tone',
    'features_resident_story',
    'campaign_name',
    'is_boosted',
]

prepublish_numeric = [
    'post_hour', 'caption_length', 'num_hashtags', 'mentions_count',
    'boost_budget_php', 'follower_count_at_post',
]

print('Output folder:', OUT_DIR)

In [ ]:
social = pd.read_csv(os.path.join(DATA_DIR, 'social_media_posts.csv'))

# Basic cleaning
social[date_col] = pd.to_datetime(social[date_col], errors='coerce', utc=True)

if label_col not in social.columns:
    raise ValueError(f"Expected label column '{label_col}' in social_media_posts.csv")

# Keep rows with a valid label and date
df = social.dropna(subset=[label_col, date_col]).copy()

df[label_col] = pd.to_numeric(df[label_col], errors='coerce')
df = df.dropna(subset=[label_col]).copy()

# Clip label to non-negative (donation value)
df[label_col] = df[label_col].clip(lower=0)

# Only keep columns we expect (ignore extras)
feature_cols = [c for c in (prepublish_categorical + prepublish_numeric) if c in df.columns]
missing = sorted(set(prepublish_categorical + prepublish_numeric) - set(feature_cols))
print('Rows:', len(df))
print('Using feature cols:', feature_cols)
print('Missing (will skip):', missing[:20], ('...' if len(missing) > 20 else ''))

# Time-based split: use latest TEST_SIZE fraction by date as holdout
cutoff = df[date_col].quantile(1 - TEST_SIZE)
train_df = df[df[date_col] <= cutoff].copy()
test_df = df[df[date_col] > cutoff].copy()

print('Train rows:', len(train_df), 'Test rows:', len(test_df))

X_train = train_df[feature_cols]
y_train = train_df[label_col]
X_test = test_df[feature_cols]
y_test = test_df[label_col]

In [ ]:
# Preprocess
numeric_features = [c for c in prepublish_numeric if c in feature_cols]
categorical_features = [c for c in prepublish_categorical if c in feature_cols]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='drop'
)

models = {
    'Ridge': Ridge(alpha=1.0, random_state=RANDOM_STATE),
    'RandomForest': RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=2,
    ),
}

rows = []
fitters = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
    mae = float(mean_absolute_error(y_test, pred))
    r2 = float(r2_score(y_test, pred))

    rows.append({'model': name, 'rmse': rmse, 'mae': mae, 'r2': r2})
    fitters[name] = pipe

results = pd.DataFrame(rows).sort_values(['mae', 'rmse']).reset_index(drop=True)
results

In [ ]:
best_name = results.loc[0, 'model']
best = fitters[best_name]

holdout_pred = best.predict(X_test)

out = test_df[['post_id', 'post_url', date_col]].copy()
out['y_true'] = y_test.values
out['y_pred'] = holdout_pred
out['abs_error'] = (out['y_true'] - out['y_pred']).abs()
out = out.sort_values('abs_error', ascending=False)

out_path = os.path.join(OUT_DIR, 'predictions.csv')
out.to_csv(out_path, index=False)

print('Best model:', best_name)
print('Wrote:', out_path)

# Small “what to do more of” table: average prediction by (platform, day_of_week, post_hour)
rec = test_df[['platform', 'day_of_week', 'post_hour']].copy()
rec['pred'] = holdout_pred
rec_table = (rec.groupby(['platform', 'day_of_week', 'post_hour'])
               .pred.mean()
               .sort_values(ascending=False)
               .head(25)
               .reset_index())
rec_table_path = os.path.join(OUT_DIR, 'recommended_windows.csv')
rec_table.to_csv(rec_table_path, index=False)
print('Wrote:', rec_table_path)

rec_table.head(10)

## Website surfacing + exported CSV schema

### Admin dashboard
- Show **predicted donation value** range for a planned post (do not present as guaranteed outcome).
- Show **top recommended windows** by platform/day/hour.

### Public impact page (public-safe)
- Prefer showing *aggregated* guidance like “best times to post” rather than per-post donation value.

### Exported files
- `ml-outputs/social-donation-lift/predictions.csv`
  - `post_id`, `post_url`, `created_at`, `y_true`, `y_pred`, `abs_error`
- `ml-outputs/social-donation-lift/recommended_windows.csv`
  - `platform`, `day_of_week`, `post_hour`, `pred` (mean predicted value in holdout)